In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from imblearn.over_sampling import SMOTE
from collections import Counter
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Create processed data folder
os.makedirs('../data/processed', exist_ok=True)

print("✅ Libraries loaded!")

In [ ]:
df = pd.read_csv('../data/creditcard.csv')
print(f"Loaded dataset: {df.shape}")
print(f"Fraud cases: {df['Class'].sum()}")
df.head()

In [ ]:
# X = all features except Class (the thing we want to predict)
# y = Class column (0 = legit, 1 = fraud)

X = df.drop('Class', axis=1)
y = df['Class']

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape:   {y.shape}")
print(f"\nFeature names: {list(X.columns)}")

In [ ]:
# V1 through V28 are already scaled (via PCA in the original dataset).
# But Amount and Time are raw — need scaling.
# We use RobustScaler (handles outliers better than StandardScaler for money data).

scaler = RobustScaler()
X['Amount_scaled'] = scaler.fit_transform(X[['Amount']])
X['Time_scaled'] = scaler.fit_transform(X[['Time']])

# Drop original unscaled versions
X = X.drop(['Amount', 'Time'], axis=1)

print("✅ Amount and Time scaled with RobustScaler")
print(f"\nAmount_scaled stats:\n{X['Amount_scaled'].describe().round(3)}")

# Save the scaler — we'll need it at prediction time to transform new transactions
joblib.dump(scaler, '../models/scaler.pkl')
print("\n💾 Scaler saved to models/scaler.pkl")

In [ ]:
# stratify=y keeps the fraud ratio (0.172%) identical in both train and test.
# Without stratification, random split might put all fraud in train or test — disaster.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20% for testing
    random_state=42,     # reproducibility — same split every time
    stratify=y           # ✨ CRITICAL: keeps class ratio intact
)

print("📊 SPLIT RESULTS")
print("=" * 50)
print(f"Training set:   {X_train.shape[0]:,} transactions")
print(f"  - Legit: {(y_train == 0).sum():,}")
print(f"  - Fraud: {(y_train == 1).sum():,} ({y_train.mean()*100:.3f}%)")
print(f"\nTest set:       {X_test.shape[0]:,} transactions")
print(f"  - Legit: {(y_test == 0).sum():,}")
print(f"  - Fraud: {(y_test == 1).sum():,} ({y_test.mean()*100:.3f}%)")
print("\n✅ Fraud % is identical in both → stratification worked")

In [ ]:
# SMOTE = Synthetic Minority Oversampling Technique
# It creates new synthetic fraud examples by interpolating between real fraud cases.
# Result: training set becomes 50/50 balanced → model actually learns fraud patterns.

# ⚠️ CRITICAL: SMOTE goes on TRAINING data ONLY, never on test data!
# Test data must stay realistic to measure real performance.

print("BEFORE SMOTE:")
print(f"  Legit: {(y_train == 0).sum():,}")
print(f"  Fraud: {(y_train == 1).sum():,}")
print(f"  Ratio: 1:{(y_train == 0).sum() // (y_train == 1).sum()}")
smote = SMOTE(random_state=42, sampling_strategy=1.0)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("\nAFTER SMOTE:")
print(f"  Legit: {(y_train_balanced == 0).sum():,}")
print(f"  Fraud: {(y_train_balanced == 1).sum():,}")
print(f"  Ratio: 1:1 ✅")
print(f"\nTotal training rows: {len(X_train_balanced):,} (was {len(X_train):,})")

In [ ]:
# Save all splits so Phases 5, 6, 7 can load them instantly without redoing this work.

joblib.dump(X_train_balanced, '../data/processed/X_train.pkl')
joblib.dump(y_train_balanced, '../data/processed/y_train.pkl')
joblib.dump(X_test, '../data/processed/X_test.pkl')
joblib.dump(y_test, '../data/processed/y_test.pkl')

# Also save a version of X_train WITHOUT SMOTE (Neural Net sometimes trains better with class_weight)
joblib.dump(X_train, '../data/processed/X_train_original.pkl')
joblib.dump(y_train, '../data/processed/y_train_original.pkl')

print("💾 All processed data saved to data/processed/")
print("\nFiles:")
for f in os.listdir('../data/processed'):
    size = os.path.getsize(f'../data/processed/{f}') / 1024**2
    print(f"  • {f}  ({size:.1f} MB)")

In [ ]:
# Quick sanity check that everything loads back correctly
X_train_loaded = joblib.load('../data/processed/X_train.pkl')
y_train_loaded = joblib.load('../data/processed/y_train.pkl')
X_test_loaded = joblib.load('../data/processed/X_test.pkl')
y_test_loaded = joblib.load('../data/processed/y_test.pkl')

print("🔍 SANITY CHECK")
print("=" * 50)
print(f"X_train shape: {X_train_loaded.shape}")
print(f"y_train class counts: {Counter(y_train_loaded)}")
print(f"\nX_test shape: {X_test_loaded.shape}")
print(f"y_test class counts: {Counter(y_test_loaded)}")
print(f"\nFeatures: {list(X_train_loaded.columns)}")
print("\n✅ Everything saved and reloadable.")
